## DATA INGESTION


In [33]:
### Document Structure

from langchain_core.documents import Document

In [34]:
doc = Document(
    page_content="This is main text content I am using to create RAG",
    metadata={
        'source':'bliss_corpus.json',
        'pages':1,
        'author':'Subodh',
        'date_created':'2026-09-22'
    }
)

print(doc)

page_content='This is main text content I am using to create RAG' metadata={'source': 'bliss_corpus.json', 'pages': 1, 'author': 'Subodh', 'date_created': '2026-09-22'}


In [35]:
### create a simple txt file
import os
os.makedirs('../data/text_files', exist_ok=True)

In [36]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for file_path, content in sample_texts.items():
    with open(file_path, 'w') as f:
        f.write(content)

print("xxxSample text files created successfully.xxx")

xxxSample text files created successfully.xxx


In [37]:
### TextLoader
#from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader

In [38]:
loader = TextLoader("../data/text_files/python_intro.txt",encoding='utf-8')
document = loader.load()
document

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]

In [39]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

# Load all text files in a directory
dir_loader=DirectoryLoader(
    "../data/text_files", 
    glob="**/*.txt", 
    loader_cls=TextLoader, 
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False
)

text_documents = dir_loader.load()
text_documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [ ]:
# LOADING JSON FILES

from langchain_community.document_loaders import JSONLoader
from pathlib import Path

file_path = Path("../data/json_files/bliss_corpus.json")

def extract_metadata(record: dict, metadata: dict) -> dict:
    metadata["doc_id"] = record.get("doc_id")
    metadata["source"] = record.get("metadata", {}).get("source")
    metadata["url"] = record.get("metadata", {}).get("url")
    metadata["topic_group"] = record.get("metadata", {}).get("topic_group")
    metadata["flags"] = record.get("metadata", {}).get("flags")
    return metadata

## JSON loader with jq schema and content key
loader = JSONLoader(
    file_path=str(file_path),
    jq_schema=".[]",  
    content_key='. | "Question: " + .question + "\nAnswer: " + .answer',
    is_content_key_jq_parsable=True,  
    metadata_func=extract_metadata,
)

documents = loader.load()
documents[:5]

[Document(metadata={'source': 'NIMH', 'seq_num': 1, 'doc_id': 'nimh_5-action-steps-to-help-someone-having-thoughts-of-suicide_01', 'url': 'https://www.nimh.nih.gov/health/publications/5-action-steps-to-help-someone-having-thoughts-of-suicide', 'topic_group': 'suicide_selfharm_crisis', 'flags': ['safety_sensitive']}, page_content='Question: If I\'m worried someone might be suicidal, is it okay to just ask them directly?\nAnswer: Yes. Directly asking someone "Are you thinking about suicide?" is the first of the 5 action steps. Research shows that asking people if they are suicidal does not increase suicidal behavior or thoughts, and the question can help open up a conversation.'),
 Document(metadata={'source': 'NIMH', 'seq_num': 2, 'doc_id': 'nimh_5-action-steps-to-help-someone-having-thoughts-of-suicide_02', 'url': 'https://www.nimh.nih.gov/health/publications/5-action-steps-to-help-someone-having-thoughts-of-suicide', 'topic_group': 'suicide_selfharm_crisis', 'flags': ['safety_sensitiv

### Creating Data Chunks

In [44]:
### Creating Data Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,  #maximum number of characters in each chunk
        chunk_overlap=chunk_overlap,  #200 characters overlap between chunks
        length_function=len,  #How to measure the length of the text
        separators=["\n\n", "\n", " ", ""]   #split hirerachy
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    #see what a chunk look like
    if split_docs:
        print(f"\nSample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  #print first 200 characters
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

chunked_documents = split_documents(text_documents, chunk_size=1000, chunk_overlap=200)
chunked_documents[:2]  #display first 2 chunks

Split 2 documents into 2 chunks.

Sample chunk:
Content: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing...
Metadata: {'source': '..\\data\\text_files\\machine_learning.txt'}


[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogr

In [ ]:
### Not Performing Chunking on Json Data because:
# each document in json dataset represents a self-contained Question & Answer pair that is already short and focused
# In RAG pipelines, chunking is designed to break down large documents (like PDFs or long articles) into 
# smaller, semantically coherent pieces so that search queries match relevant paragraphs instead of huge walls of text.

### Embeddings and VectorStore DB